# Objective: Train and compare multiple classification models to predict the quality score of wine (typically a scale of 3–8) based on its physicochemical properties such as acidity, density, and alcohol content. 

### Tech Stack: Python, pandas, numpy, scikit-learn (Random Forest, SGD, SVC), seaborn, matplotlib, Jupyter Notebook 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

In [ ]:
df=pd.read_csv('data\WineQT.csv')

In [ ]:
df.head()

In [ ]:
#rows and columns
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#checking null values
df.isnull().sum()

No null values!

In [ ]:
quality_counts = df["quality"].value_counts().sort_index()

quality_counts

In [ ]:
#checking class distribution
plt.figure(figsize=(8,5))

sns.countplot(x="quality", data=df)

plt.title("Distribution of Wine Quality Scores")
plt.savefig('images/wine_distribution.png')
plt.show()

# Class Imbalance

The quality scores are not evenly distributed.

Most wines belong to the middle quality categories (5 and 6), while very low and very high quality wines occur much less frequently.
This imbalance can affect model training because machine learning algorithms tend to learn the majority classes better than minority classes.
To reduce this problem and create a more practical prediction task, the quality scores are converted into binary classes representing **Good** and **Bad** wine.

## Observations

- The dataset contains multiple physicochemical measurements.
- The target variable is **quality**.
- No major missing values are observed.
- Some quality scores appear much more frequently than others.

In [ ]:
# removing the id column
df.drop("Id", axis=1, inplace=True)

### EDA plots

In [ ]:
df.hist(figsize=(18,15), bins=20)

plt.tight_layout()
plt.savefig('images/plots.png')
plt.show()

In [ ]:
plt.figure(figsize=(12,10))

sns.heatmap(df.corr(),
            annot=True,
            cmap="coolwarm",
            fmt=".2f")

plt.title("Correlation Heatmap")
plt.savefig('images/correlation_heatmap.png')
plt.show()

The distribution plots help identify the spread, skewness, and potential outliers in each feature.
The correlation heatmap highlights relationships between chemical properties and wine quality.
Features such as alcohol, sulphates, and volatile acidity generally exhibit stronger relationships with wine quality compared to others.

### Feature Engineering

In [ ]:
df["quality"] = df["quality"].apply(lambda x: 1 if x >= 6 else 0)
df["quality"].value_counts()

In [ ]:
sns.countplot(x="quality", data=df)
plt.savefig('images/quality_distr_binary.png')
plt.show()

The original quality scores were grouped into two categories:

- 0 → Bad Quality (quality < 6)
- 1 → Good Quality (quality ≥ 6)

Binary classification simplifies the prediction task, reduces class imbalance, and makes the model more practical for real-world applications where wines are often categorized as acceptable or not acceptable.

### Split features

In [ ]:
X = df.drop("quality", axis=1)
y = df["quality"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Data Preparation

The dataset is divided into training and testing sets using an 80:20 ratio.
Stratified sampling is used to preserve the proportion of each quality class in both datasets.
Feature scaling is performed using StandardScaler for algorithms such as SGD and SVC, which are sensitive to differences in feature scales.

### Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

### SGD (Stochastic Gradient Descent)

In [ ]:
sgd = SGDClassifier(random_state=42)

sgd.fit(X_train_scaled, y_train)
sgd_pred = sgd.predict(X_test_scaled)

### SVC (Support Vector Classifier)

In [ ]:
svc = SVC(random_state=42)

svc.fit(X_train_scaled, y_train)
svc_pred = svc.predict(X_test_scaled)

### Evalation

In [ ]:
def evaluate(y_true, pred):

    print("Accuracy:", accuracy_score(y_true, pred))
    print()
    print(classification_report(y_true, pred))

    cm = confusion_matrix(y_true, pred)
    sns.heatmap(cm,
                annot=True,
                fmt="d",
                cmap="Blues")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
#random forest classiier
evaluate(y_test, rf_pred)
plt.savefig('images/rf_heatmap.png')

In [ ]:
#sgd evaluation
evaluate(y_test,sgd_pred)
plt.savefig('images/sgd_heatmap.png')

In [ ]:
#svc evaluation
evaluate(y_test,svc_pred)
plt.savefig('images/svc_heatmap.png')

# Model Comparison

The performance of the three classification models was evaluated using accuracy, precision, recall, and F1-score.

| Model | Accuracy | Precision | Recall | F1-score |
|--------|----------|-----------|--------|----------|
| Random Forest | **80.35%** | **80%** | **85%** | **82%** |
| Support Vector Classifier (SVC) | 78.60% | **80%** | 81% | 80% |
| Stochastic Gradient Descent (SGD) | 62.88% | 62% | 84% | 71% |

### Observations

- **Random Forest** achieved the highest overall accuracy (80.35%) and the best F1-score (82%), demonstrating the most balanced performance across both classes.
- **Support Vector Classifier (SVC)** also performed well, with an accuracy of 78.60%. Its precision matched Random Forest, but its recall and F1-score were slightly lower.
- **SGD Classifier** achieved a relatively high recall (84%), meaning it identified most good-quality wines. However, its low precision (62%) indicates that many wines predicted as good were actually of poor quality, resulting in a higher number of false positives.
- Since this project does not specify that false positives or false negatives are more costly than the other, both precision and recall are equally important. Therefore, the **F1-score**, which balances these two metrics, provides a more reliable measure of model performance than either metric alone.

### Feature importance

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

importance.sort_values().plot(
    kind="barh",
    figsize=(8,6),
    color="darkcyan"
)

plt.title("Random Forest Feature Importance")
plt.savefig('images/feature_imp.png')
plt.show()

### Observations

- **Alcohol** is the most influential feature, indicating that alcohol content plays the largest role in predicting wine quality.
- **Sulphates**, **total sulfur dioxide**, and **volatile acidity** are also important predictors, suggesting that these chemical properties significantly contribute to distinguishing good-quality wines from bad-quality wines.
- **Density** has a moderate influence on the prediction, while features such as **fixed acidity**, **chlorides**, and **pH** contribute to a lesser extent.
- **Residual sugar**, **citric acid**, and **free sulfur dioxide** have the lowest importance scores among the features, indicating that they have comparatively less impact on the model's decisions.
- Although some features have lower individual importance, they still contribute to the overall predictive performance when combined with other physicochemical properties.

Overall, the feature importance analysis shows that wine quality is influenced by a combination of chemical characteristics rather than a single factor, with **alcohol content emerging as the strongest predictor** in the Random Forest model.

### Comparison

In [ ]:
comparison= pd.DataFrame({
    'Model':['Random forest','SGD','SVC'],

    'Accuracy': [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, sgd_pred),
        accuracy_score(y_test, svc_pred)
    ]
})
comparison.sort_values(by='Accuracy', ascending=False)


In [ ]:
plt.figure(figsize=(7,5))

sns.barplot(
    data=comparison,
    x="Model",
    y="Accuracy"
)

plt.ylim(0,1)

plt.title("Model Accuracy Comparison")
plt.savefig('images/model_comparison.png')
plt.show()

### Observations

- **Random Forest** achieved the highest accuracy (**80.35%**), making it the best-performing model for this classification task.
- **Support Vector Classifier (SVC)** performed competitively with an accuracy of **78.60%**, only slightly lower than Random Forest.
- **SGD Classifier** achieved the lowest accuracy (**62.88%**), indicating that it was less effective at distinguishing between good- and bad-quality wines.
- The superior performance of Random Forest can be attributed to its ability to capture complex, non-linear relationships among the physicochemical properties of wine while being robust to noise and overfitting.
- Considering the evaluation metrics (accuracy, precision, recall, and F1-score), **Random Forest demonstrated the most balanced and reliable performance**, making it the most suitable model for deployment.

# Conclusion

In this project, three machine learning classification models—**Random Forest**, **Support Vector Classifier (SVC)**, and **Stochastic Gradient Descent (SGD)**—were developed and evaluated to predict wine quality using its physicochemical properties.

The dataset was first explored through exploratory data analysis (EDA), which revealed the distribution of chemical features and the relationships between them. Since the original quality scores were imbalanced, they were converted into two categories (Good and Bad) to simplify the classification task and improve model performance. A stratified train-test split was used to preserve the class distribution during training and testing.

Among the three models, **Random Forest achieved the best overall performance**, with an accuracy of **80.35%**, outperforming both SVC (78.60%) and SGD (62.88%). It also provided the highest F1-score, demonstrating a strong balance between precision and recall. Furthermore, Random Forest offered feature importance scores, making the model more interpretable by identifying the chemical properties that contribute most to wine quality prediction.

The feature importance analysis showed that **alcohol** was the most influential predictor, followed by **sulphates**, **total sulfur dioxide**, and **volatile acidity**, indicating that these characteristics play a significant role in determining wine quality.

Overall, **Random Forest is the most suitable model for deployment** because it delivers the highest predictive accuracy, maintains a balanced performance across evaluation metrics, handles non-linear relationships effectively, and provides interpretable feature importance. Future improvements could include hyperparameter tuning, cross-validation, and addressing class imbalance using resampling techniques to further enhance model performance.